# Hindustani Raga Classifier — trained from scratch on the TRF dataset

**Why this notebook exists**: this project's fused similarity vector used to include a `raga_probability` feature that turned out to be fake (a hand-typed template heuristic, never a real classifier). Research into real alternatives found the [DeepSRGM](https://github.com/shubhlohiya/automatic-raga-recognition) pretrained checkpoint, but it only classifies 10 **Carnatic** ragas — the wrong tradition for a Bollywood/ghazal/sufi catalog. The [Thaat and Raga Forest (TRF) dataset](https://www.kaggle.com/datasets/suryamajumder/thaat-and-raga-forest-trf-dataset) is **Hindustani** tradition (10 thaats, 61 ragas — Bhairavi, Yaman, Khamaj, Malkauns, Desh, Bageshree, Darbari, Pahadi... exactly the ragas Bollywood/ghazal composers actually draw on) and explicitly includes movie songs, not just concert recordings. But it has **no pretrained weights at all** — its own reference repo is training-code-only. So: train one ourselves.

**This is a personal-learning exercise**, not a production integration — the dataset is AGPL-3.0 licensed (fine for personal use; relevant if this project is ever redistributed or run as a network service).

**Before running:**
1. Add Data → search `suryamajumder/thaat-and-raga-forest-trf-dataset` → Add. It's ~19GB; Kaggle mounts it read-only at `/kaggle/input/`, no download needed on your end.
2. Notebook Settings → Accelerator → **GPU T4 x2**. Internet isn't strictly required (everything needed is preinstalled), but leaving it On doesn't hurt.
3. This trains for real — expect it to take a real fraction of your free weekly GPU quota. Progress prints per epoch so you can judge and stop early if needed.

**Output**: a trained PyTorch model (`.pt`), the raga/thaat label list, the exact preprocessing config, and an honest held-out test accuracy report (both per-segment and per-file majority-vote, matching how it'll actually be used — classifying a whole song, not one random clip) — staged under `/kaggle/working/` for `kaggle kernels output` to pull down afterward, same as the main batch runner notebook.

In [ ]:
# --- Config ---
SAMPLE_RATE = 22050
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
SEGMENT_SECONDS = 30          # long enough for a raga's melodic phrase to develop
MIN_TEST_SEGMENT_SECONDS = 15 # drop a trailing test/val chunk shorter than this

BATCH_SIZE = 32
EPOCHS = 25
LR = 1e-3
EARLY_STOP_PATIENCE = 6
MAX_TRAIN_SECONDS = 8 * 3600  # safety net against a free-tier session getting killed mid-epoch

SEED = 42
NUM_WORKERS = 2  # DataLoader workers — matches this project's established RAM-conscious default

In [ ]:
import subprocess, time

def run(cmd, timeout, label=None):
    label = label or cmd
    print(f"--- RUNNING ({timeout}s timeout): {label}")
    t0 = time.time()
    try:
        result = subprocess.run(cmd, shell=True, timeout=timeout,
                                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    except subprocess.TimeoutExpired as e:
        print(e.stdout or "")
        raise RuntimeError(f"TIMED OUT after {time.time()-t0:.0f}s: {label}")
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"FAILED (exit {result.returncode}): {label}")
    print(f"--- OK ({time.time()-t0:.0f}s): {label}")
    return result

# mp3 decoding backend — cheap, already proven fast on Kaggle in the main batch runner.
run("apt-get -qq install -y ffmpeg", timeout=120, label="apt-get ffmpeg")

In [ ]:
# --- Locate the dataset under /kaggle/input (don't hardcode the exact mount path) ---
import glob, os

candidates = glob.glob("/kaggle/input/**/Thaat and Raga Forest*", recursive=True)
candidates = [c for c in candidates if os.path.isdir(c)]
if not candidates:
    raise RuntimeError(
        "TRF dataset not found under /kaggle/input. Add it: Add Data > search "
        "'suryamajumder/thaat-and-raga-forest-trf-dataset' > Add."
    )
DATASET_ROOT = candidates[0]
print("Dataset root:", DATASET_ROOT)

In [ ]:
# --- Enumerate every recording, labeled by (thaat, raga) from its folder path ---
from collections import defaultdict

files_by_raga = defaultdict(list)
for path in glob.glob(os.path.join(DATASET_ROOT, "*", "*", "*.mp3")):
    parts = path.split(os.sep)
    thaat, raga = parts[-3], parts[-2]
    files_by_raga[(thaat, raga)].append(path)

ragas = sorted(files_by_raga.keys(), key=lambda k: k[1])
raga2idx = {raga_key: i for i, raga_key in enumerate(ragas)}
idx2raga = {i: {"thaat": t, "raga": r} for (t, r), i in raga2idx.items()}

total_files = sum(len(v) for v in files_by_raga.values())
print(f"{len(ragas)} ragas, {total_files} recordings total")
for (t, r), fs in sorted(files_by_raga.items(), key=lambda kv: kv[0][1]):
    print(f"  {t:20s} {r:35s} n={len(fs)}")

In [ ]:
# --- File-level train/val/test split (never split a recording's segments across sets) ---
import random
random.seed(SEED)

train_files, val_files, test_files = [], [], []
for raga_key, fs in files_by_raga.items():
    fs = sorted(fs)
    random.shuffle(fs)
    label = raga2idx[raga_key]
    if len(fs) >= 3:
        test_files.append((fs[0], label))
        val_files.append((fs[1], label))
        train_files.extend((f, label) for f in fs[2:])
    else:
        # too few recordings for a held-out split — train-only, flagged so the
        # final report doesn't quietly present train-only classes as validated.
        train_files.extend((f, label) for f in fs)

train_only_ragas = [idx2raga[i]["raga"] for i in set(raga2idx.values())
                     if i not in {l for _, l in val_files} and i not in {l for _, l in test_files}]
print(f"train={len(train_files)}  val={len(val_files)}  test={len(test_files)}")
if train_only_ragas:
    print(f"Ragas with <3 recordings (train-only, no held-out evaluation): {train_only_ragas}")

In [ ]:
# --- Dataset: decode one segment per __getitem__ via librosa's offset/duration seek,
# not the whole file — these recordings run 5-15+ minutes each. ---
import numpy as np
import librosa
import torch
from torch.utils.data import Dataset

def to_logmel(y):
    mel = librosa.feature.melspectrogram(y=y, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
    return librosa.power_to_db(mel, ref=np.max).astype(np.float32)

class RagaSegmentDataset(Dataset):
    """Random 30s crop per recording per access (fresh each epoch — a form of
    free data augmentation given these are long recordings)."""
    def __init__(self, file_label_pairs):
        self.files = file_label_pairs
        self._durations = {}

    def _duration(self, path):
        if path not in self._durations:
            self._durations[path] = librosa.get_duration(path=path)
        return self._durations[path]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path, label = self.files[idx]
        dur = self._duration(path)
        max_start = max(0.0, dur - SEGMENT_SECONDS)
        start = random.uniform(0, max_start) if max_start > 0 else 0.0
        y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True, offset=start, duration=SEGMENT_SECONDS)
        needed = int(SEGMENT_SECONDS * SAMPLE_RATE)
        if len(y) < needed:
            y = np.pad(y, (0, needed - len(y)))
        return torch.from_numpy(to_logmel(y)).unsqueeze(0), label

def segment_file(path, segment_seconds=SEGMENT_SECONDS, min_seconds=MIN_TEST_SEGMENT_SECONDS):
    """Split a full recording into non-overlapping segments for evaluation —
    used at test time so we can majority-vote across the whole song, matching
    how this model will actually be used later (classify a whole track)."""
    dur = librosa.get_duration(path=path)
    starts = np.arange(0, dur, segment_seconds)
    segs = []
    for s in starts:
        if dur - s < min_seconds:
            continue
        y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True, offset=float(s), duration=segment_seconds)
        needed = int(segment_seconds * SAMPLE_RATE)
        if len(y) < needed:
            y = np.pad(y, (0, needed - len(y)))
        segs.append(to_logmel(y))
    return segs

In [ ]:
# --- Model: a compact CNN over log-mel spectrograms. AdaptiveAvgPool2d at the
# end means it doesn't care about exact time-axis length. ---
import torch.nn as nn

class RagaCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )
        self.features = nn.Sequential(
            block(1, 32), block(32, 64), block(64, 128), block(128, 256),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type != "cuda":
    raise RuntimeError("No GPU detected — training this on CPU would take far too long. Check Notebook Settings > Accelerator.")

In [ ]:
# --- Training setup: class-weighted loss to handle the 5-42 files/raga imbalance ---
from torch.utils.data import DataLoader
from sklearn.utils.class_weight import compute_class_weight

train_labels = np.array([label for _, label in train_files])
class_weights_arr = compute_class_weight("balanced", classes=np.arange(len(ragas)), y=train_labels)
class_weights = torch.tensor(class_weights_arr, dtype=torch.float32).to(device)

train_loader = DataLoader(RagaSegmentDataset(train_files), batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)
val_loader = DataLoader(RagaSegmentDataset(val_files), batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS)

model = RagaCNN(num_classes=len(ragas)).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

print(f"train batches/epoch: {len(train_loader)}  val batches: {len(val_loader)}")

In [ ]:
# --- Training loop ---
best_val_acc = 0.0
epochs_no_improve = 0
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
        train_correct += (out.argmax(1) == y).sum().item()
        train_total += x.size(0)

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            val_correct += (out.argmax(1) == y).sum().item()
            val_total += x.size(0)

    train_acc = train_correct / max(1, train_total)
    val_acc = val_correct / max(1, val_total)
    scheduler.step(val_acc)
    elapsed = time.time() - t_start
    print(f"epoch {epoch:2d}/{EPOCHS}  train_loss={train_loss/train_total:.4f}  "
          f"train_acc={train_acc:.3f}  val_acc={val_acc:.3f}  elapsed={elapsed/60:.1f}min")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), "/kaggle/working/raga_model_best.pt")
        print(f"  -> new best (val_acc={val_acc:.3f}), checkpoint saved")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"No val improvement for {EARLY_STOP_PATIENCE} epochs — stopping early.")
            break

    if elapsed > MAX_TRAIN_SECONDS:
        print(f"Hit MAX_TRAIN_SECONDS ({MAX_TRAIN_SECONDS}s) safety limit — stopping.")
        break

print(f"Best val accuracy: {best_val_acc:.3f}")

In [ ]:
# --- Held-out TEST evaluation: per-segment AND per-file majority-vote accuracy.
# Majority-vote is the number that actually matters for the real use case
# (classifying a whole song), matching DeepSRGM's own evaluation methodology. ---
model.load_state_dict(torch.load("/kaggle/working/raga_model_best.pt"))
model.eval()

per_class_correct = defaultdict(int)
per_class_total = defaultdict(int)
seg_correct, seg_total = 0, 0
file_correct, file_total = 0, 0

with torch.no_grad():
    for path, label in test_files:
        segs = segment_file(path)
        if not segs:
            continue
        x = torch.from_numpy(np.stack(segs)).unsqueeze(1).to(device)
        preds = model(x).argmax(1).cpu().numpy()
        seg_correct += int((preds == label).sum())
        seg_total += len(preds)
        majority = int(np.bincount(preds).argmax())
        file_total += 1
        per_class_total[label] += 1
        if majority == label:
            file_correct += 1
            per_class_correct[label] += 1

print(f"Per-segment test accuracy: {seg_correct/max(1,seg_total):.3f}  ({seg_correct}/{seg_total})")
print(f"Per-file majority-vote test accuracy: {file_correct/max(1,file_total):.3f}  ({file_correct}/{file_total})")
print()
print("Per-class (majority-vote) breakdown:")
for label in sorted(per_class_total):
    n = per_class_total[label]
    c = per_class_correct[label]
    print(f"  {idx2raga[label]['raga']:35s} {c}/{n}")

In [ ]:
# --- Save artifacts for download (kaggle kernels output) and later integration ---
import json

with open("/kaggle/working/raga_classes.json", "w") as f:
    json.dump(idx2raga, f, indent=2, ensure_ascii=False)

with open("/kaggle/working/preprocessing_config.json", "w") as f:
    json.dump({
        "sample_rate": SAMPLE_RATE, "n_mels": N_MELS, "n_fft": N_FFT,
        "hop_length": HOP_LENGTH, "segment_seconds": SEGMENT_SECONDS,
    }, f, indent=2)

with open("/kaggle/working/results.json", "w") as f:
    json.dump({
        "best_val_acc": best_val_acc,
        "test_segment_acc": seg_correct / max(1, seg_total),
        "test_majority_vote_acc": file_correct / max(1, file_total),
        "num_ragas": len(ragas),
        "train_only_ragas_no_holdout": train_only_ragas,
    }, f, indent=2)

print("Saved: raga_model_best.pt, raga_classes.json, preprocessing_config.json, results.json")
print("Click 'Save Version' to publish these as a Kaggle Dataset version.")